In [50]:
import pandas as pd
import webbrowser
# Import the required module from the fyers_apiv3 package
from fyers_apiv3 import fyersModel
from datetime import datetime, timedelta
import pandas as pd
import time

# Configuration

In [51]:
resolution= 3
ticker = "NSE:AXISBANK-EQ"
start_date = datetime(2024, 1, 1)
# end_date = datetime(2026, 1, 9)
end_date = datetime.now()
resolution = 3

#### Create Auth Token

In [52]:
def generate_fyers_access_token(
    client_id: str,
    secret_key: str,
    redirect_uri: str,
    state: str = "sample_state"
) -> dict:
    """
    Handles FYERS OAuth flow and returns access token response.

    Returns
    -------
    dict
        Full response containing access_token, refresh_token, expiry, etc.
    """

    # -------------------------------
    # Step 1: Generate auth URL
    # -------------------------------
    session = fyersModel.SessionModel(
        client_id=client_id,
        secret_key=secret_key,
        redirect_uri=redirect_uri,
        response_type="code",
        state=state
    )

    auth_url = session.generate_authcode()
    webbrowser.open(auth_url)

    # -------------------------------
    # Step 2: User pastes redirected URL
    # -------------------------------
    raw_code = input("Paste redirected URL here: ").strip()

    try:
        auth_code = raw_code.split("auth_code=")[1].split("&")[0]
    except IndexError:
        raise ValueError("Invalid redirect URL. Auth code not found.")

    # -------------------------------
    # Step 3: Exchange auth_code for access_token
    # -------------------------------
    session = fyersModel.SessionModel(
        client_id=client_id,
        secret_key=secret_key,
        redirect_uri=redirect_uri,
        response_type="code",
        grant_type="authorization_code"
    )

    session.set_token(auth_code)
    token_response = session.generate_token()

    return token_response


In [53]:
client_id="ALT6RUE1IF-100"
secret_key="0UT0LW5PE4"
redirect_uri="https://luvratan.tech/"

token_response = generate_fyers_access_token(
    client_id=client_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri
)

access_token = token_response["access_token"]
print("Access Token:", access_token)


Access Token: eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJhdWQiOlsiZDoxIiwiZDoyIiwieDowIiwieDoxIl0sImF0X2hhc2giOiJnQUFBQUFCcGRrdkxqSDFJSDQtUDJNZnRkeXVRX3NSUDdseDNtQ1pDXzhWMmdzQ2JUYUc4WW9hLWRPa3hFRUktbWpKeDRRblBjcFJTalBaQjI3MjRwdjBBYlpwdGxHTXB1Mm1CamtDMU84SHVmNjJqVWJDMkYtWT0iLCJkaXNwbGF5X25hbWUiOiIiLCJvbXMiOiJLMSIsImhzbV9rZXkiOiIzN2RhNWQ2OGVhOTQ2YzQyODJiMGI0NjMwMDgyZTQ2YzA2ZDMyY2I2NTllZTczMTdkZTVlODc2OCIsImlzRGRwaUVuYWJsZWQiOiJOIiwiaXNNdGZFbmFibGVkIjoiTiIsImZ5X2lkIjoiWEwwMDcyMSIsImFwcFR5cGUiOjEwMCwiZXhwIjoxNzY5Mzg3NDAwLCJpYXQiOjE3NjkzNjAzMzEsImlzcyI6ImFwaS5meWVycy5pbiIsIm5iZiI6MTc2OTM2MDMzMSwic3ViIjoiYWNjZXNzX3Rva2VuIn0.5XDj7H8OZMWMTuzg4YtQDySbxWM0XRFWKxdXoDhmLko


In [54]:
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token, log_path="")

data = {
    "symbol":"NSE:SBIN-EQ",
    "resolution":"15",
    "date_format":"1",
    "range_from":"2018-01-01",
    "range_to":"2018-03-31",
    "cont_flag":"1"
}

response = fyers.history(data=data)
print(response)



{'candles': [[1514778300, 310.6, 311.9, 310.2, 310.9, 513902], [1514779200, 310.8, 312.7, 310.75, 311.85, 604528], [1514780100, 311.85, 311.95, 311.25, 311.45, 205501], [1514781000, 311.45, 311.95, 311.1, 311.2, 162179], [1514781900, 311.15, 311.5, 311.05, 311.3, 116801], [1514782800, 311.3, 311.8, 311.2, 311.55, 161985], [1514783700, 311.55, 311.75, 311.2, 311.25, 1006953], [1514784600, 311.25, 311.3, 310.55, 310.6, 300691], [1514785500, 310.6, 310.8, 310.4, 310.7, 292488], [1514786400, 310.7, 310.8, 310.35, 310.5, 159422], [1514787300, 310.5, 310.7, 310.3, 310.4, 135650], [1514788200, 310.4, 310.4, 309.7, 309.95, 401549], [1514789100, 309.95, 310.25, 309.9, 310.2, 190149], [1514790000, 310.25, 310.3, 309.9, 310.2, 290368], [1514790900, 310.2, 310.2, 309.5, 309.7, 311588], [1514791800, 309.75, 309.9, 309.15, 309.2, 313604], [1514792700, 309.2, 309.4, 308.65, 308.9, 389269], [1514793600, 308.9, 309.05, 308.8, 308.85, 198064], [1514794500, 308.85, 309, 308.1, 308.6, 326514], [1514795400

#### Get Data

In [55]:
import pandas as pd

def fyers_history_to_df(response: dict) -> pd.DataFrame:
    """
    Convert FYERS history API response to a pandas DataFrame.
    Shifts intraday candle timestamps forward by 15 minutes.
    """

    if response.get("s") != "ok":
        raise ValueError(f"FYERS API error: {response}")
    print(response["candles"])
    df = pd.DataFrame(
        response["candles"],
        columns=["timestamp", "open", "high", "low", "close", "volume"]
    )

    # Convert UNIX epoch → timezone-aware datetime (IST)
    df["datetime"] = (
        pd.to_datetime(df["timestamp"], unit="s", utc=True)
        .dt.tz_convert("Asia/Kolkata")
    )

    # Shift candle time forward by 15 minutes
    df["datetime"] = df["datetime"] + pd.Timedelta(minutes=resolution)

    # Sort by adjusted time
    df = df.sort_values("datetime")

    # Drop raw timestamp if not needed
    df = df.drop(columns=["timestamp"])

    return df


df = fyers_history_to_df(response)
df

[[1514778300, 310.6, 311.9, 310.2, 310.9, 513902], [1514779200, 310.8, 312.7, 310.75, 311.85, 604528], [1514780100, 311.85, 311.95, 311.25, 311.45, 205501], [1514781000, 311.45, 311.95, 311.1, 311.2, 162179], [1514781900, 311.15, 311.5, 311.05, 311.3, 116801], [1514782800, 311.3, 311.8, 311.2, 311.55, 161985], [1514783700, 311.55, 311.75, 311.2, 311.25, 1006953], [1514784600, 311.25, 311.3, 310.55, 310.6, 300691], [1514785500, 310.6, 310.8, 310.4, 310.7, 292488], [1514786400, 310.7, 310.8, 310.35, 310.5, 159422], [1514787300, 310.5, 310.7, 310.3, 310.4, 135650], [1514788200, 310.4, 310.4, 309.7, 309.95, 401549], [1514789100, 309.95, 310.25, 309.9, 310.2, 190149], [1514790000, 310.25, 310.3, 309.9, 310.2, 290368], [1514790900, 310.2, 310.2, 309.5, 309.7, 311588], [1514791800, 309.75, 309.9, 309.15, 309.2, 313604], [1514792700, 309.2, 309.4, 308.65, 308.9, 389269], [1514793600, 308.9, 309.05, 308.8, 308.85, 198064], [1514794500, 308.85, 309, 308.1, 308.6, 326514], [1514795400, 308.65, 30

,open,high,low,close,volume,datetime
0,310.60,311.90,310.20,310.90,513902,2018-01-01 09:18:00+05:30
1,310.80,312.70,310.75,311.85,604528,2018-01-01 09:33:00+05:30
2,311.85,311.95,311.25,311.45,205501,2018-01-01 09:48:00+05:30
3,311.45,311.95,311.10,311.20,162179,2018-01-01 10:03:00+05:30
4,311.15,311.50,311.05,311.30,116801,2018-01-01 10:18:00+05:30
...,...,...,...,...,...,...
1491,251.60,252.35,251.30,251.80,587239,2018-03-28 14:18:00+05:30
1492,251.80,251.85,250.55,250.60,732483,2018-03-28 14:33:00+05:30
1493,250.75,252.45,250.45,251.95,1253492,2018-03-28 14:48:00+05:30
1494,251.90,252.10,249.00,250.35,7257425,2018-03-28 15:03:00+05:30


In [56]:
def combined_data_function(ticker="NSE:SBIN-EQ", resolution=15, start_date="2018-01-01", end_date="2018-03-31"):
    fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token, log_path="")

    data = {
        "symbol": ticker, #"NSE:SBIN-EQ",
        "resolution":resolution, #"15",
        "date_format":"1",
        "range_from":start_date, #"2018-01-01",
        "range_to":end_date, #"2018-03-31",
        "cont_flag":"1",
        "oi_flag":1
    }

    response = fyers.history(data=data)
    df = fyers_history_to_df(response)
    df = df.set_index("datetime")

    return df


In [57]:
# ticker = "NSE:RELIANCE-EQ"
# df = combined_data_function("NSE:RELIANCE-EQ", resolution=5, start_date="2025-10-01", end_date="2025-12-31")
# df.head()

In [58]:
# file_name= ticker.split(":")[1].split("-")[0]
# df.to_csv(f"./5_minutes_data/{file_name}.csv", index=None)

#### Fetch Multiple Years for any minutes interval

In [59]:




# Generate 3-month intervals
date_ranges = []
current_start = start_date

while current_start < end_date:
    # Calculate end of 3-month period
    current_end = current_start + timedelta(days=90)
    
    # Don't exceed the final end date
    if current_end > end_date:
        current_end = end_date
    
    date_ranges.append({
        'start': current_start.strftime('%Y-%m-%d'),
        'end': current_end.strftime('%Y-%m-%d')
    })
    
    # Move to next period (start from day after current end)
    current_start = current_end + timedelta(days=1)

print(f"Total periods to fetch: {len(date_ranges)}")

# Fetch and append data
all_data = []

for i, period in enumerate(date_ranges):
    print(f"Fetching period {i+1}/{len(date_ranges)}: {period['start']} to {period['end']}")
    
    try:
        df = combined_data_function(
            ticker=ticker,
            resolution=resolution,
            start_date=period['start'],
            end_date=period['end']
        )
        
        all_data.append(df)
        print(f"  ✓ Fetched {len(df)} rows")
        
        # Add delay to avoid rate limiting
        time.sleep(1)
        
    except Exception as e:
        print(f"  ✗ Error fetching data: {e}")
        continue

# Combine all dataframes
if all_data:
    combined_df = pd.concat(all_data, ignore_index=False)
    combined_df = combined_df.sort_index()
    
    # Remove any duplicate timestamps
    combined_df = combined_df[~combined_df.index.duplicated(keep='first')]
    
    print(f"\nTotal rows collected: {len(combined_df)}")
    print(f"Date range: {combined_df.index.min()} to {combined_df.index.max()}")
    
    # Save to CSV
    file_name = ticker.split(":")[1].split("-")[0]
    # combined_df.to_csv(output_path)
    
    # print(f"✓ Saved to {output_path}")
else:
    print("No data collected")

Total periods to fetch: 9
Fetching period 1/9: 2024-01-01 to 2024-03-31
[[1704080700, 1095, 1099, 1093.65, 1096, 110612], [1704080880, 1096.15, 1096.9, 1095, 1095.7, 156639], [1704081060, 1095.6, 1096.45, 1095.3, 1095.45, 26830], [1704081240, 1095.3, 1097.8, 1095.3, 1096.55, 21177], [1704081420, 1096.5, 1099.1, 1096.5, 1098, 63645], [1704081600, 1097.65, 1098.6, 1097.1, 1097.95, 10701], [1704081780, 1097.9, 1098.6, 1097.7, 1098.3, 14406], [1704081960, 1098.3, 1099.5, 1097.7, 1099.4, 25943], [1704082140, 1099.4, 1100.55, 1098.9, 1099.6, 26324], [1704082320, 1099.6, 1100.35, 1098, 1098.2, 21044], [1704082500, 1098.2, 1099, 1098, 1098.85, 21536], [1704082680, 1098.8, 1099.65, 1098.5, 1099.6, 10112], [1704082860, 1099.6, 1100.4, 1099, 1100.4, 11125], [1704083040, 1100.45, 1100.8, 1100, 1100.6, 26515], [1704083220, 1100.6, 1100.9, 1100, 1100, 24573], [1704083400, 1100.1, 1100.45, 1100, 1100.05, 8281], [1704083580, 1100.45, 1100.7, 1099.9, 1100.35, 9135], [1704083760, 1100.35, 1101.6, 1099.9

In [60]:
# output_path = f"./5_min_data/{file_name}.csv"
output_path = f"./3_min_data/{file_name}.csv"
# output_path = f"./1_hour_data/{file_name}.csv"

combined_df.to_csv(output_path)

##### Close

In [61]:
# len(ticker_list)

In [62]:
# import pandas as pd
# from datetime import datetime
# from dateutil.relativedelta import relativedelta
# import os

# def fetch_and_save_fyers_data(
#     ticker="NSE:BSE-EQ",
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-04",
#     save_dir="./Data/15_min_data"
# ):
#     """
#     Fetch FYERS historical data in 3-month chunks, concatenate, and save as CSV.
#     """

#     # Initialize FYERS
#     fyers = fyersModel.FyersModel(
#         client_id=client_id,
#         is_async=False,
#         token=access_token,
#         log_path=""
#     )

#     start_dt = datetime.strptime(start_date, "%Y-%m-%d")
#     end_dt = datetime.strptime(end_date, "%Y-%m-%d")

#     dfs = []
#     curr_start = start_dt

#     while curr_start <= end_dt:
#         curr_end = min(curr_start + relativedelta(months=3) - relativedelta(days=1), end_dt)

#         data = {
#             "symbol": ticker,
#             "resolution": str(resolution),
#             "date_format": "1",
#             "range_from": curr_start.strftime("%Y-%m-%d"),
#             "range_to": curr_end.strftime("%Y-%m-%d"),
#             "cont_flag": "1"
#         }

#         print(f"Fetching: {data['range_from']} → {data['range_to']}")

#         response = fyers.history(data=data)

#         if response.get("candles"):
#             df = fyers_history_to_df(response)
#             df["datetime"] = pd.to_datetime(df["datetime"])
#             dfs.append(df)
#         else:
#             print(f"No data returned for {data['range_from']} → {data['range_to']}")

#         curr_start = curr_end + relativedelta(days=1)

#     if not dfs:
#         raise ValueError("No data fetched for the given date range.")

#     # Concatenate once (important for performance)
#     final_df = (
#         pd.concat(dfs, ignore_index=True)
#           .drop_duplicates(subset=["datetime"])
#           .sort_values("datetime")
#           .set_index("datetime")
#     )

#     # Prepare filename
#     os.makedirs(save_dir, exist_ok=True)
#     symbol_name = ticker.replace(":", "_").replace("-", "_").lower()
#     filepath = os.path.join(save_dir, f"{symbol_name}.csv")

#     final_df.to_csv(filepath)
#     print(f"Saved data → {filepath}")

#     return final_df
# fetch_and_save_fyers_data(
#     ticker="NSE:SBIN-EQ",
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-09",
#     save_dir="15_min_data"
# )

In [63]:
# for ticker in ticker_list:
#     fetch_and_save_fyers_data(
#     ticker=ticker,
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-04",
#     save_dir="15_min_data"
#     )

In [64]:
# def fetch_and_save_fyers_data_worker(args):
#     return fetch_and_save_fyers_data(**args)


#### Batch Processing

In [65]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# import time
# from pathlib import Path
# import json
# from datetime import datetime

# def parallel_fetch_fyers(
#     ticker_list,
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-04",
#     save_dir="./Data/15_min_data",
#     max_workers=3,
#     rate_limit_delay=0.5
# ):
#     """
#     Parallel historical fetch with rate limiting and progress tracking.
#     """
    
#     tasks = [
#         {
#             "ticker": ticker,
#             "resolution": resolution,
#             "start_date": start_date,
#             "end_date": end_date,
#             "save_dir": save_dir
#         }
#         for ticker in ticker_list
#     ]

#     results = {"success": [], "failed": [], "errors": {}}
    
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         # Submit with slight delays to avoid rate limit spikes
#         futures = {}
#         for i, task in enumerate(tasks):
#             future = executor.submit(fetch_and_save_fyers_data_worker, task)
#             futures[future] = task["ticker"]
#             if i < len(tasks) - 1:
#                 time.sleep(rate_limit_delay)
        
#         # Track progress
#         start_time = time.time()
#         for i, future in enumerate(as_completed(futures), 1):
#             ticker = futures[future]
#             elapsed = time.time() - start_time
#             avg_time = elapsed / i
#             eta = avg_time * (len(tasks) - i)
            
#             try:
#                 future.result()
#                 results["success"].append(ticker)
#                 print(f"✅ [{i}/{len(tasks)}] {ticker} | ETA: {eta/60:.1f}m")
#             except Exception as e:
#                 error_msg = str(e)[:200]
#                 results["failed"].append(ticker)
#                 results["errors"][ticker] = error_msg
#                 print(f"❌ [{i}/{len(tasks)}] {ticker} | Error: {error_msg}")
    
#     return results


# def batch_fetch_fyers(
#     ticker_list,
#     batch_size=20,
#     max_retries=2,
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-04",
#     save_dir="./Data/15_min_data",
#     max_workers=3,
#     rate_limit_delay=0.5,
#     save_progress=True
# ):
#     """
#     Fetch 100+ tickers in batches with retry logic and progress saving.
#     """
    
#     print(f"🚀 Starting batch fetch for {len(ticker_list)} tickers")
#     print(f"   Batch size: {batch_size}, Workers: {max_workers}")
#     print(f"   Rate limit delay: {rate_limit_delay}s")
#     print("="*60)
    
#     all_results = {
#         "success": [],
#         "failed": [],
#         "errors": {},
#         "metadata": {
#             "total_tickers": len(ticker_list),
#             "start_time": datetime.now().isoformat(),
#             "resolution": resolution,
#             "date_range": f"{start_date} to {end_date}"
#         }
#     }
    
#     # Process in batches
#     num_batches = (len(ticker_list) + batch_size - 1) // batch_size
    
#     for batch_num in range(num_batches):
#         start_idx = batch_num * batch_size
#         end_idx = min(start_idx + batch_size, len(ticker_list))
#         batch = ticker_list[start_idx:end_idx]
        
#         print(f"\n📦 BATCH {batch_num + 1}/{num_batches}")
#         print(f"   Tickers {start_idx + 1}-{end_idx} of {len(ticker_list)}")
#         print(f"   Tickers: {', '.join(batch[:5])}{'...' if len(batch) > 5 else ''}")
#         print("-"*60)
        
#         # Fetch this batch
#         batch_results = parallel_fetch_fyers(
#             ticker_list=batch,
#             resolution=resolution,
#             start_date=start_date,
#             end_date=end_date,
#             save_dir=save_dir,
#             max_workers=max_workers,
#             rate_limit_delay=rate_limit_delay
#         )
        
#         # Aggregate results
#         all_results["success"].extend(batch_results["success"])
#         all_results["failed"].extend(batch_results["failed"])
#         all_results["errors"].update(batch_results["errors"])
        
#         # Save progress checkpoint
#         if save_progress:
#             progress_file = Path(save_dir) / "fetch_progress.json"
#             with open(progress_file, 'w') as f:
#                 json.dump(all_results, f, indent=2)
        
#         # Cooldown between batches (except last batch)
#         if batch_num < num_batches - 1:
#             cooldown = 3
#             print(f"\n💤 Cooldown {cooldown}s before next batch...")
#             time.sleep(cooldown)
    
#     # Retry failed tickers
#     if all_results["failed"] and max_retries > 0:
#         print(f"\n{'='*60}")
#         print(f"🔄 RETRY PHASE: {len(all_results['failed'])} failed tickers")
#         print(f"   Retries remaining: {max_retries}")
#         print("="*60)
        
#         for retry_num in range(max_retries):
#             if not all_results["failed"]:
#                 break
            
#             failed_tickers = all_results["failed"].copy()
#             all_results["failed"] = []
            
#             print(f"\n🔄 Retry {retry_num + 1}/{max_retries}")
            
#             retry_results = parallel_fetch_fyers(
#                 ticker_list=failed_tickers,
#                 resolution=resolution,
#                 start_date=start_date,
#                 end_date=end_date,
#                 save_dir=save_dir,
#                 max_workers=max(1, max_workers - 1),  # More conservative
#                 rate_limit_delay=rate_limit_delay * 1.5
#             )
            
#             all_results["success"].extend(retry_results["success"])
#             all_results["failed"] = retry_results["failed"]
#             all_results["errors"].update(retry_results["errors"])
            
#             if retry_num < max_retries - 1 and all_results["failed"]:
#                 time.sleep(5)  # Longer cooldown between retries
    
#     # Final summary
#     all_results["metadata"]["end_time"] = datetime.now().isoformat()
#     print_final_summary(all_results, save_dir)
    
#     # Save final results
#     if save_progress:
#         final_file = Path(save_dir) / f"fetch_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
#         with open(final_file, 'w') as f:
#             json.dump(all_results, f, indent=2)
#         print(f"\n💾 Results saved to: {final_file}")
    
#     return all_results


# def print_final_summary(results, save_dir):
#     """Print comprehensive summary of fetch operation."""
#     total = results["metadata"]["total_tickers"]
#     success_count = len(results["success"])
#     failed_count = len(results["failed"])
#     success_rate = (success_count / total * 100) if total > 0 else 0
    
#     print(f"\n{'='*60}")
#     print("📊 FINAL SUMMARY")
#     print("="*60)
#     print(f"✅ Success: {success_count}/{total} ({success_rate:.1f}%)")
#     print(f"❌ Failed:  {failed_count}/{total}")
    
#     if results["failed"]:
#         print(f"\n⚠️  Failed tickers ({len(results['failed'])}):")
#         for ticker in results["failed"][:10]:
#             error = results["errors"].get(ticker, "Unknown error")
#             print(f"   • {ticker}: {error[:80]}")
#         if len(results["failed"]) > 10:
#             print(f"   ... and {len(results['failed']) - 10} more")
    
#     # Check what files were actually created
#     saved_files = list(Path(save_dir).glob("*.csv"))
#     print(f"\n💾 Files saved: {len(saved_files)} CSV files in {save_dir}")
    
#     print("="*60)


# def resume_failed_fetch(results_file, **kwargs):
#     """Resume fetch from saved progress file."""
#     with open(results_file, 'r') as f:
#         previous_results = json.load(f)
    
#     failed_tickers = previous_results.get("failed", [])
    
#     if not failed_tickers:
#         print("✅ No failed tickers to resume!")
#         return previous_results
    
#     print(f"🔄 Resuming {len(failed_tickers)} failed tickers from {results_file}")
    
#     return batch_fetch_fyers(
#         ticker_list=failed_tickers,
#         **kwargs
#     )

In [66]:

# # Run with optimal settings for BSE/NSE
# results = batch_fetch_fyers(
#     ticker_list=ticker_list,
#     batch_size=20,           # Process 20 tickers per batch
#     max_retries=2,           # Retry failed tickers twice
#     resolution=15,
#     start_date="2018-01-01",
#     end_date="2026-01-04",
#     save_dir="./Data/15_min_data",
#     max_workers=3,           # 3 concurrent requests
#     rate_limit_delay=0.5,    # 0.5s between request submissions
#     save_progress=True       # Save checkpoint after each batch
# )